# Session 3 Lab: CampusBuddy, a Chatbot with Memory

**Project 2 · Chatbot with memory** · GenAI & Agentic AI Workshop

You'll call an LLM from Python, check what each answer costs, stream answers, and build a chatbot that remembers the conversation.

**Before you start**
1. Get a free Gemini API key at [Google AI Studio](https://aistudio.google.com/apikey).
2. In Colab, click the 🔑 **Secrets** icon on the left, add a secret named `GEMINI_API_KEY`, paste your key and switch on *Notebook access*.
3. Run each cell with **Shift + Enter**.

⚠️ Never paste your key directly into the code. If you push this notebook to GitHub, the key goes with it.

In [ ]:
!pip install -q google-genai

## Step 1: Your first call

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"   # any current free model from Google AI Studio works

resp = client.models.generate_content(
    model=MODEL,
    contents="Explain what a compiler does in 2 lines, for a first-year student.",
)
print(resp.text)

## Step 2: What did that cost?

Every response tells you how many tokens went in and came out.

In [ ]:
u = resp.usage_metadata
print("Input tokens: ", u.prompt_token_count)
print("Output tokens:", u.candidates_token_count)
print("Total tokens: ", u.total_token_count)
print("Finish reason:", resp.candidates[0].finish_reason)

In [ ]:
# Illustrative prices in $ per million tokens. Check the provider's pricing page for real ones.
PRICE_IN, PRICE_OUT = 0.30, 2.50

def cost(resp):
    u = resp.usage_metadata
    out = (u.candidates_token_count or 0) + (getattr(u, "thoughts_token_count", 0) or 0)  # thinking tokens are billed as output
    return (u.prompt_token_count * PRICE_IN + out * PRICE_OUT) / 1_000_000

print(f"That answer cost ${cost(resp):.6f}")
print(f"1 lakh answers like it would cost about ${cost(resp) * 100_000:.2f}")

## Step 3: Settings

Add a system prompt, temperature and a length limit. Try changing `temperature` and `max_output_tokens` and run again.

In [ ]:
resp = client.models.generate_content(
    model=MODEL,
    contents="Suggest 3 names for our college coding club.",
    config=types.GenerateContentConfig(
        system_instruction="You are a creative assistant. Reply as a numbered list.",
        temperature=1.0,
        max_output_tokens=200,
    ),
)
print(resp.text)

## Step 4: Streaming

Watch the answer appear piece by piece.

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Give 5 tips to prepare for campus placements.",
):
    print(chunk.text or "", end="", flush=True)

## Step 5: A chatbot with memory

The model has **no memory**. We keep the conversation in a list called `history` and send the whole list every time.

In [ ]:
SYSTEM = """You are "CampusBuddy", a friendly helpdesk assistant for ABC Engineering College.
- Help with questions about exams, fees, hostel, library and placements.
- Keep answers under 80 words, in simple English.
- If you don't know something about the college, say so. Don't make it up."""

history = []

def chat(user_text):
    history.append({"role": "user", "parts": [{"text": user_text}]})
    resp = client.models.generate_content(
        model=MODEL,
        contents=history,                    # send EVERYTHING so far
        config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.5),
    )
    history.append({"role": "model", "parts": [{"text": resp.text}]})
    return resp

print(chat("Hi, I'm Priya from CSE, 2nd year.").text)
print(chat("What's my name, branch and year?").text)

## Step 6: Break the memory (on purpose)

This version sends **only the new message**. Ask it your name. What happens?

In [ ]:
def chat_no_memory(user_text):
    resp = client.models.generate_content(
        model=MODEL,
        contents=user_text,                  # only the new message
        config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.5),
    )
    return resp.text

print(chat_no_memory("Hi, I'm Priya from CSE, 2nd year."))
print(chat_no_memory("What's my name, branch and year?"))

## Step 7: The chat loop

Type your questions. Type `quit` to stop. After each answer you'll see the tokens and cost. Watch the **input tokens** grow every turn: that's the history being re-sent.

In [ ]:
history = []          # start a fresh conversation
total = 0.0

while True:
    q = input("You: ").strip()
    if q.lower() in ("quit", "exit", "bye", ""):
        break
    r = chat(q)
    total += cost(r)
    u = r.usage_metadata
    print("CampusBuddy:", r.text)
    print(f"   [in {u.prompt_token_count} tokens, out {u.candidates_token_count} tokens, "
          f"this turn ${cost(r):.6f}, total ${total:.6f}]\n")

print(f"Chat ended after {len(history) // 2} turns. Total cost ${total:.6f}")

## Stretch 1: Keep only the last 6 messages

Change `contents=history` to `contents=history[-6:]` inside `chat()`, run the loop again, and chat for 6+ turns. Do the input tokens stop growing? What does the bot forget?

## Stretch 2: Retry when you hit the rate limit

In [ ]:
import time
from google.genai import errors

def ask_with_retry(prompt, tries=3):
    for i in range(tries):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt).text
        except errors.APIError as e:
            if e.code in (429, 500, 503) and i < tries - 1:
                wait = 5 * 2 ** i            # 5s, then 10s, then 20s
                print(f"Error {e.code}, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

print(ask_with_retry("Say 'retry works' in 3 words."))

## Stretch 3: Same bot on Groq

Get a free key at [console.groq.com](https://console.groq.com), add a secret named `GROQ_API_KEY`, and run this. Notice the role is `assistant`, not `model`, and the system prompt is the first message.

In [ ]:
!pip install -q groq
from groq import Groq

groq = Groq(api_key=userdata.get("GROQ_API_KEY"))
messages = [{"role": "system", "content": SYSTEM}]

def chat_groq(user_text):
    messages.append({"role": "user", "content": user_text})
    r = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",     # check console.groq.com for current models
        messages=messages,
    )
    reply = r.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat_groq("Hi, I'm Priya from CSE."))
print(chat_groq("What's my name?"))

## Done? 🎉

You built a chatbot with memory, streaming and cost tracking. Save a copy (**File → Save a copy in Drive**) and push it to your GitHub as Project 2. Make sure no API key is written in the notebook first!

**Next session:** we give CampusBuddy your college's real documents with RAG, so it can answer from facts instead of guessing.